<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/Analytics-Forecast/blob/main/Train_Overall_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

In [14]:
def load_data(file_path):
    df = pd.read_csv(file_path)

    data_df = df.copy()

    data_df['Date'] = pd.to_datetime(
        data_df['Date'],
        dayfirst=True
    )

    data_df = data_df.sort_values(
        ['BranchID', 'Date']
    )

    return data_df

In [15]:
TARGET_COLUMNS = [
    'CountofGroup',
    'CountofClientName',
    'CountofAccountID',
    'CountofCreditOfficerID',
    'SumofPrincipalOutstanding',
    'SumofInterestOutstanding',
    'SumofTotalPrincipalOverdue',
    'SumofTotalInterestOverdue',
    'SumofTotalPAR'
]

FEATURES = [
    'lag_1',
    'lag_2',
    'ema_7',
    'lag_3',
    'diff_1'
]


def create_time_series_features(
    data_df,
    target_columns,
    group_col='BranchID'
):

    for col in target_columns:
        for lag in [1, 2, 3]:
            data_df[f'{col}_lag_{lag}'] = (
                data_df
                .groupby(group_col)[col]
                .shift(lag)
            )

        data_df[f'{col}_ema_7'] = (
            data_df
            .groupby(group_col)[col]
            .transform(
                lambda x: x.shift(1).ewm(span=7).mean()
            )
        )

        data_df[f'{col}_diff_1'] = (
            data_df
            .groupby(group_col)[col]
            .diff(1)
        )

    return data_df

In [16]:
df = load_data('/content/dat.csv')
df = create_time_series_features(
    df,
    TARGET_COLUMNS
)